# Day 12 / 42: How ML Actually Works
### 42 Days of ML Challenge | @VaishnaviJagtap18

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week2_feature_work/day12_how_ml_works/day12_notebook.ipynb)

---

## What You Will Learn
- What a model actually IS: a function with adjustable numbers (parameters)
- What "training" means in concrete terms: search for the best parameters
- The loss function: how the model knows if it's wrong
- Gradient Descent: built from scratch, no sklearn, to see exactly what `.fit()` does
- Proof that hand-built gradient descent converges to the SAME answer as sklearn
- How learning rate controls whether training succeeds or fails

---

## Step 0: Install and Import

In [ ]:
!pip install numpy matplotlib scikit-learn --quiet

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful. You are ready for Day 12.")

---
## Step 1: What Is a Model, Really?

Strip away the jargon. A linear regression model is this:

$$\hat{y} = w \cdot x + b$$

- **x** is your input (square footage of a house)
- **w** is a number called a **weight** (how much price increases per square foot)
- **b** is a number called a **bias** (the baseline price even at 0 sq ft)
- **ŷ** (y-hat) is the prediction

**"Training a model" means finding the values of w and b that make predictions match reality as closely as possible.**

That's it. There is no other magic. Every model — from linear regression to a 70-billion-parameter LLM — is a function with adjustable numbers, and training is the search for the best values of those numbers.

In [ ]:
# Build a house price dataset
np.random.seed(42)
n = 300

sqft  = np.random.normal(1500, 400, n)          # square footage
price = 50 + 0.12 * sqft + np.random.normal(0, 25, n)   # price in lakhs (INR)

print("Sample data (sqft -> price in lakhs):")
for i in range(5):
    print(f"  {sqft[i]:>7.0f} sqft  ->  {price[i]:>6.1f} lakhs")

print(f"\nDataset size: {n} houses")
print(f"sqft range: {sqft.min():.0f} to {sqft.max():.0f}")
print(f"price range: {price.min():.1f} to {price.max():.1f} lakhs")

plt.figure(figsize=(7,5))
plt.scatter(sqft, price, alpha=0.5, color='steelblue', edgecolor='white')
plt.xlabel('Square Footage')
plt.ylabel('Price (lakhs INR)')
plt.title('House Price Data')
plt.tight_layout()
plt.savefig('day12_data.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 2: The Loss Function — How the Model Knows It's Wrong

Pick ANY values for w and b. You get a prediction line. Some predictions will be close to the real prices, some far off.

**Mean Squared Error (MSE)** measures how wrong the model is on average:

$$\text{Loss} = \frac{1}{m}\sum_{i=1}^{m}(\hat{y}_i - y_i)^2$$

- Take the difference between prediction and actual price for every house
- Square it (so negative and positive errors don't cancel out, and big errors are punished more)
- Average across all houses

**Lower loss = better model. Training = making this number as small as possible.**

Let's compute the loss for a few random guesses of w and b, before any "training" happens.

In [ ]:
def compute_loss(w, b, X, y):
    """Mean Squared Error for predictions w*X + b against true y"""
    y_hat = w * X + b
    return np.mean((y_hat - y) ** 2)

# Try 4 random guesses for (w, b)
guesses = [
    (0.0, 0.0),    # predicts 0 for every house — terrible
    (0.5, 50.0),   # way too steep
    (0.05, 100.0), # too flat, too high baseline
    (0.12, 50.0),  # close to the real relationship
]

print(f"{'w':>6} {'b':>6} | {'Loss (MSE)':>12}")
print("-" * 30)
for w, b in guesses:
    loss = compute_loss(w, b, sqft, price)
    print(f"{w:>6.2f} {b:>6.1f} | {loss:>12.1f}")

print()
print("Notice: as (w,b) gets closer to the true relationship (price = 0.12*sqft + 50),")
print("the loss drops dramatically. This number IS the signal the model optimizes.")

---
## Step 3: Visualizing the Loss Surface

Every combination of (w, b) produces a loss value. Plot loss against all combinations of w and b, and you get a **3D bowl shape**.

Training a model is literally finding the bottom of this bowl.

(We normalize sqft first — this is the scaling concept from Day 11. Without it, the bowl becomes a long thin valley and training is much slower.)

In [ ]:
# Normalize sqft (mean=0, std=1) — makes the loss surface a clean bowl
sqft_norm = (sqft - sqft.mean()) / sqft.std()

# Build a grid of (w, b) values and compute loss at each point
w_range = np.linspace(0, 90, 50)
b_range = np.linspace(150, 310, 50)
W, B = np.meshgrid(w_range, b_range)

L = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        L[i, j] = compute_loss(W[i, j], B[i, j], sqft_norm, price)

min_idx = np.unravel_index(L.argmin(), L.shape)
print(f"Lowest loss in this grid: {L.min():.1f}")
print(f"At w = {W[min_idx]:.1f}, b = {B[min_idx]:.1f}")
print()
print("This is the BOTTOM OF THE BOWL — the best (w,b) combination.")
print("Gradient Descent is the algorithm that finds this point automatically.")

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, L, cmap='viridis', alpha=0.85)
ax.scatter([W[min_idx]], [B[min_idx]], [L.min()], color='red', s=80, label='Minimum loss')
ax.set_xlabel('w (weight)')
ax.set_ylabel('b (bias)')
ax.set_zlabel('Loss (MSE)')
ax.set_title('The Loss Surface — Training = Finding the Bottom of This Bowl')
plt.tight_layout()
plt.savefig('day12_loss_surface.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Step 4: Gradient Descent — Built From Scratch

You can't search every possible (w, b) combination — there are infinitely many.

**Gradient Descent** is the algorithm that finds the bottom of the bowl efficiently:

1. Start with random w and b (usually 0, 0)
2. Compute the **gradient** — the slope of the loss surface at the current point. This tells you which direction is "downhill"
3. Take a small step downhill: `w = w - learning_rate * gradient`
4. Repeat thousands of times until the loss stops decreasing

The gradients for MSE loss are:

$$\frac{\partial L}{\partial w} = \frac{2}{m}\sum (\hat{y}_i - y_i) \cdot x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{m}\sum (\hat{y}_i - y_i)$$

**This is what `.fit()` does inside sklearn.** Below, we write it ourselves — no library shortcuts.

In [ ]:
def gradient_descent(X, y, learning_rate=0.1, epochs=100):
    """
    Train y = w*X + b using gradient descent.
    Returns final w, b, and the loss at every epoch.
    """
    w, b = 0.0, 0.0   # start at the origin — no idea yet
    m = len(X)
    loss_history = []

    for epoch in range(epochs):
        # Step 1: make predictions with current w, b
        y_hat = w * X + b

        # Step 2: compute loss (just to track progress)
        loss = np.mean((y_hat - y) ** 2)
        loss_history.append(loss)

        # Step 3: compute gradients — direction of steepest INCREASE in loss
        dw = (2/m) * np.sum((y_hat - y) * X)
        db = (2/m) * np.sum(y_hat - y)

        # Step 4: step in the OPPOSITE direction (downhill)
        w = w - learning_rate * dw
        b = b - learning_rate * db

    return w, b, loss_history


# Train on normalized sqft
w_final, b_final, history = gradient_descent(sqft_norm, price, learning_rate=0.1, epochs=100)

print("=== Gradient Descent Results ===")
print(f"Final w: {w_final:.4f}")
print(f"Final b: {b_final:.4f}")
print(f"Loss at epoch 0:   {history[0]:.2f}")
print(f"Loss at epoch 100: {history[-1]:.2f}")
print()
print(f"Loss dropped by {history[0] - history[-1]:.1f} points over 100 epochs.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: loss decreasing over epochs
axes[0].plot(history, color='crimson', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Decreasing During Training')
axes[0].grid(alpha=0.3)

# Right: the fitted line on the data
axes[1].scatter(sqft_norm, price, alpha=0.4, color='steelblue', label='Actual data')
x_line = np.linspace(sqft_norm.min(), sqft_norm.max(), 100)
y_line = w_final * x_line + b_final
axes[1].plot(x_line, y_line, color='red', linewidth=2.5, label='Fitted line (from scratch GD)')
axes[1].set_xlabel('Square Footage (normalized)')
axes[1].set_ylabel('Price (lakhs)')
axes[1].set_title('What Gradient Descent Found')
axes[1].legend()

plt.tight_layout()
plt.savefig('day12_gd_training.png', dpi=120, bbox_inches='tight')
plt.show()

print("This red line is the result of 100 small steps downhill.")
print("Nobody told the algorithm the answer. It found this by following the slope.")

---
## Step 5: Proof — From-Scratch GD Matches sklearn Exactly

If our understanding of `.fit()` is correct, our hand-built gradient descent should land on the SAME w and b as sklearn's `LinearRegression`.

This is the moment the black box opens.

In [ ]:
# sklearn's LinearRegression on the same normalized data
sklearn_model = LinearRegression()
sklearn_model.fit(sqft_norm.reshape(-1, 1), price)

w_sklearn = sklearn_model.coef_[0]
b_sklearn = sklearn_model.intercept_

print("=== From-Scratch Gradient Descent ===")
print(f"  w = {w_final:.6f}")
print(f"  b = {b_final:.6f}")
print()
print("=== sklearn LinearRegression.fit() ===")
print(f"  w = {w_sklearn:.6f}")
print(f"  b = {b_sklearn:.6f}")
print()
print(f"Difference in w: {abs(w_final - w_sklearn):.6f}")
print(f"Difference in b: {abs(b_final - b_sklearn):.6f}")
print()
print("sklearn's .fit() runs the exact same loop you wrote above.")
print("It's just written in optimized C/Cython and runs more iterations.")
print("Now you know what happens when you call .fit() on ANY model.")

---
## Step 6: Learning Rate — The Knob That Can Break Everything

The **learning rate** controls how big each step downhill is.

- **Too small**: training crawls. After 100 epochs you're barely closer to the answer.
- **Just right**: steady, fast convergence to the minimum.
- **Too large**: each step overshoots the bowl, bounces to the other side, and gets WORSE. Loss explodes to infinity.

Let's see all three happen with real numbers.

In [ ]:
def gradient_descent_safe(X, y, learning_rate, epochs=50):
    """Same as before, but stops early if loss explodes (diverges)."""
    w, b = 0.0, 0.0
    m = len(X)
    history = []
    for epoch in range(epochs):
        y_hat = w * X + b
        loss = np.mean((y_hat - y) ** 2)
        history.append(loss)
        if np.isnan(loss) or loss > 1e8:
            break
        dw = (2/m) * np.sum((y_hat - y) * X)
        db = (2/m) * np.sum(y_hat - y)
        w -= learning_rate * dw
        b -= learning_rate * db
    return history

learning_rates = {
    'Too Small (0.001)': 0.001,
    'Just Right (0.1)':  0.1,
    'Too Large (1.5)':   1.5,
}

print(f"{'Learning Rate':<20} | {'Loss Start':>12} | {'Loss End':>15} | {'Epochs Run':>10}")
print("-" * 65)

results = {}
for label, lr in learning_rates.items():
    hist = gradient_descent_safe(sqft_norm, price, lr, epochs=50)
    results[label] = hist
    end_loss = hist[-1]
    end_str = f"{end_loss:.1f}" if end_loss < 1e8 else "EXPLODED (inf)"
    print(f"{label:<20} | {hist[0]:>12.1f} | {end_str:>15} | {len(hist):>10}")

fig, ax = plt.subplots(figsize=(8,5))
for label, hist in results.items():
    clipped = np.clip(hist, 0, 2000)  # clip for visibility
    ax.plot(clipped, label=label, linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (clipped at 2000 for visibility)')
ax.set_title('Effect of Learning Rate on Training')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('day12_learning_rates.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print("lr=0.001: loss barely moves. Would need thousands of epochs.")
print("lr=0.1: smooth, fast convergence.")
print("lr=1.5: loss explodes to infinity within a few steps. Training has DIVERGED.")

---
## Step 7: The Real-World Production Problem

**The scenario:** An MLE trains a neural network. Loss starts at 4.2, drops to 3.9, then suddenly becomes `nan` (Not a Number) by epoch 3. The model is now useless — every weight is `nan`.

The team spends a day checking for bad data, missing values, corrupted labels. Nothing is wrong with the data.

**The actual cause:** the learning rate was too high for this dataset. The first few steps overshot the minimum so badly that the weights became enormous numbers, then `inf`, then `nan` once arithmetic overflowed.

**The fix:** lower the learning rate by 10x (a common starting move is trying 0.1, 0.01, 0.001 and picking the largest one that doesn't diverge). This single hyperparameter is responsible for more "my model isn't learning" debugging sessions than almost anything else.

Below: reproduce the `nan` failure and the fix, using the numbers from Step 6.

In [ ]:
# Reproduce: training that diverges to nan
broken_history = gradient_descent_safe(sqft_norm, price, learning_rate=2.5, epochs=10)

print("=== Training with learning_rate = 2.5 ===")
for i, loss in enumerate(broken_history):
    status = "OK" if loss < 1e6 else "EXPLODING"
    print(f"  Epoch {i}: loss = {loss:.2f}   [{status}]")

print()
print("=== Same model, learning_rate = 0.1 (10x smaller) ===")
fixed_history = gradient_descent_safe(sqft_norm, price, learning_rate=0.1, epochs=10)
for i, loss in enumerate(fixed_history):
    print(f"  Epoch {i}: loss = {loss:.2f}")

print()
print("Same data. Same model. Same starting point. Only the learning rate changed.")
print("This is the single most common reason a model 'doesn't train.'")

---
## Step 8: Summary — The Learn-From-Data Loop

In [ ]:
print("=" * 62)
print("DAY 12 SUMMARY: How ML Actually Works")
print("=" * 62)
print()
print("THE LOOP, EVERY MODEL FOLLOWS THIS:")
print("-" * 50)
print("1. Model = a function with adjustable numbers (parameters)")
print("   Linear regression: y_hat = w*x + b")
print("   Neural network: thousands/millions of w's and b's")
print()
print("2. Make predictions with current parameters")
print("3. Measure how wrong those predictions are (the LOSS)")
print("4. Compute the GRADIENT — direction that makes loss worse")
print("5. Step in the OPPOSITE direction (GRADIENT DESCENT)")
print("6. Repeat steps 2-5 thousands of times")
print()
print("KEY RULES")
print("-" * 50)
print("7. Lower loss = better model. That's the entire objective.")
print("8. Learning rate too small -> training crawls")
print("9. Learning rate too large -> loss explodes to nan/inf")
print("10. .fit() in sklearn/PyTorch/TensorFlow runs exactly this loop")
print("11. Scaling your features (Day 11) makes the loss surface a")
print("    clean bowl instead of a long thin valley -> faster convergence")
print()
print("=" * 62)

---
## Practice Exercise

A new dataset is given below: years of experience vs salary (in lakhs).

Your tasks:
1. Normalize the experience values
2. Write your own `compute_loss(w, b, X, y)` function (copy from Step 2 if needed)
3. Implement gradient descent from scratch and train for 200 epochs with `learning_rate=0.1`
4. Print the final w and b
5. Compare your result to sklearn's `LinearRegression` — they should match closely
6. Try `learning_rate=2.0` and observe what happens to the loss

In [ ]:
# Practice dataset — experience vs salary
np.random.seed(7)
n_practice = 200

experience = np.random.uniform(0, 15, n_practice)            # years
salary = 4 + 1.8 * experience + np.random.normal(0, 1.5, n_practice)  # lakhs

print("Sample data (experience -> salary):")
for i in range(5):
    print(f"  {experience[i]:>5.1f} yrs  ->  {salary[i]:>5.1f} lakhs")

print(f"\nDataset size: {n_practice}")
print()
print("Your tasks:")
print("  1. Normalize 'experience'")
print("  2. Write compute_loss(w, b, X, y)")
print("  3. Train with gradient descent, lr=0.1, 200 epochs")
print("  4. Print final w, b")
print("  5. Compare to sklearn LinearRegression")
print("  6. Try lr=2.0 — what happens?")

# --- Your solution below ---


In [ ]:
# SOLUTION — try on your own first before looking here

exp_norm = (experience - experience.mean()) / experience.std()

def compute_loss_practice(w, b, X, y):
    y_hat = w * X + b
    return np.mean((y_hat - y) ** 2)

def gd_practice(X, y, lr, epochs):
    w, b = 0.0, 0.0
    m = len(X)
    history = []
    for _ in range(epochs):
        y_hat = w * X + b
        loss = compute_loss_practice(w, b, X, y)
        history.append(loss)
        if loss > 1e8 or np.isnan(loss):
            break
        dw = (2/m) * np.sum((y_hat - y) * X)
        db = (2/m) * np.sum(y_hat - y)
        w -= lr * dw
        b -= lr * db
    return w, b, history

# Train with lr=0.1
w_p, b_p, hist_p = gd_practice(exp_norm, salary, lr=0.1, epochs=200)
print(f"From-scratch GD (lr=0.1, 200 epochs): w={w_p:.4f}, b={b_p:.4f}")
print(f"Loss: {hist_p[0]:.2f} -> {hist_p[-1]:.2f}")

# Compare to sklearn
sk_model = LinearRegression().fit(exp_norm.reshape(-1,1), salary)
print(f"sklearn LinearRegression:             w={sk_model.coef_[0]:.4f}, b={sk_model.intercept_:.4f}")

# Try lr=2.0
_, _, hist_bad = gd_practice(exp_norm, salary, lr=2.0, epochs=10)
print(f"\nWith lr=2.0, loss after 10 epochs: {hist_bad[-1]:.2e}")
print("Loss exploded — the learning rate was too large for this loss surface.")

---
## What's Next

**Day 13: Linear Regression**  
Now that you've built gradient descent from scratch, Day 13 goes deeper into linear regression itself — the cost function, coefficients, R² and RMSE, and how Zillow uses it as a baseline before complex models.

---
**GitHub repo:** https://github.com/VaishnaviJagtap18/-42-Days-of-ML-Challenge  
**LinkedIn:** Follow for Day 13 tomorrow  
#42DaysOfML #MachineLearning #Python #MLEngineer